# Data Preprocessing

Following the exploratory data analysis, the dataset is now prepared for machine learning.

The preprocessing stage will focus on:

1. Creating a clean working copy of the dataset.
2. Removing duplicate records.
3. Converting and validating the date variable.
4. Handling missing values.
5. Checking numerical feature ranges and invalid values.
6. Preparing the target variable.
7. Separating predictors and target.
8. Preparing a leakage-safe preprocessing pipeline for model development.

The original dataset will not be modified. All preprocessing operations will be performed on a working copy.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# ============================================================
# LOAD MESOGEOS DATASET
# ============================================================

DATASET_PATH = Path("../../dataset/mesogeos_wildfire_dataset.csv")

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (11305, 32)


In [3]:
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())


Columns:
['date', 'latitude', 'longitude', 'temperature_c', 'dew_point_c', 'relative_humidity', 'wind_speed', 'wind_direction', 'rainfall_mm', 'surface_pressure', 'solar_radiation', 'ndvi', 'lai', 'soil_moisture', 'elevation', 'slope_degrees', 'aspect', 'curvature', 'roads_distance_km', 'population', 'lc_agriculture', 'lc_forest', 'lc_grassland', 'lc_settlement', 'lc_shrubland', 'lc_sparse_vegetation', 'lc_water_bodies', 'lc_wetland', 'burned_area_ha', 'year', 'month', 'day_of_year']

First 5 records:


,date,latitude,longitude,temperature_c,dew_point_c,relative_humidity,wind_speed,wind_direction,rainfall_mm,surface_pressure,...,lc_grassland,lc_settlement,lc_shrubland,lc_sparse_vegetation,lc_water_bodies,lc_wetland,burned_area_ha,year,month,day_of_year
0,2006-08-27,38.123648,-7.819233,36.419031,15.989252,22.510025,3.511700,329.388733,0.000000,99694.617188,...,0.0,0.0,0.000000,0.0,0.0,0.0,40.0,2006,8,239
1,2006-08-11,42.000202,0.447392,27.559045,7.369226,16.027991,6.567619,296.081879,0.004796,93547.523438,...,0.0,0.0,0.192824,0.0,0.0,0.0,1390.0,2006,8,223
2,2006-07-11,41.617581,-8.524061,34.471796,20.400690,35.828364,2.905688,345.955994,0.550396,99709.054688,...,0.0,0.0,0.000000,0.0,0.0,0.0,173.0,2006,7,192
3,2006-09-07,41.557167,-7.567509,30.666193,12.939386,28.009099,2.133366,313.057190,2.396404,93731.859375,...,0.0,0.0,0.000000,0.0,0.0,0.0,220.0,2006,9,250
4,2006-09-08,41.547098,-6.248474,31.690332,14.382166,22.408468,3.914824,288.407166,0.974770,93459.937500,...,0.0,0.0,0.000000,0.0,0.0,0.0,76.0,2006,9,251


In [4]:
df_clean = df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

Original dataset shape: (11305, 32)
Working dataset shape: (11305, 32)


## 1. Remove Duplicate Records

The exploratory data analysis identified two completely duplicated records.

Duplicate observations can give repeated events disproportionate influence during model training. Therefore, the duplicated records will be removed from the working dataset.

The original CSV file will remain unchanged.

In [5]:
duplicates_before = df_clean.duplicated().sum()

print("Duplicate records before removal:", duplicates_before)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

duplicates_after = df_clean.duplicated().sum()

print("Duplicate records after removal:", duplicates_after)
print("Clean dataset shape:", df_clean.shape)

Duplicate records before removal: 2
Duplicate records after removal: 0
Clean dataset shape: (11303, 32)


In [6]:
assert df_clean.duplicated().sum() == 0

print("Duplicate verification passed.")
print("Final unique records:", len(df_clean))

Duplicate verification passed.
Final unique records: 11303


## 2. Validate and Convert the Date Variable

The `date` variable is currently stored as a string. It will be converted to Pandas datetime format so that temporal features can be reliably extracted.

The conversion will also be used to identify invalid or incorrectly formatted dates.

The existing `year`, `month`, and `day_of_year` variables will subsequently be checked against the converted date to ensure temporal consistency.

In [7]:
print("Date data type:", df_clean["date"].dtype)

print("\nSample date values:")
display(df_clean["date"].head(10))

print("\nNumber of unique date values:", df_clean["date"].nunique())

Date data type: str

Sample date values:


0    2006-08-27
1    2006-08-11
2    2006-07-11
3    2006-09-07
4    2006-09-08
5    2006-07-11
6    2006-08-30
7    2006-09-04
8    2006-09-03
9    2006-06-21
Name: date, dtype: str


Number of unique date values: 2819


In [8]:
# CONVERT DATE TO DATETIME

df_clean["date"] = pd.to_datetime(
    df_clean["date"],
    errors="coerce"
)

print("Date data type after conversion:", df_clean["date"].dtype)

print(
    "\nInvalid dates created by conversion:",
    df_clean["date"].isna().sum()
)

Date data type after conversion: datetime64[us]

Invalid dates created by conversion: 0


In [9]:
# DATE RANGE CHECK

print("Earliest date:", df_clean["date"].min())
print("Latest date:", df_clean["date"].max())

print("\nNumber of records by year:")
display(
    df_clean["date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Earliest date: 2006-04-09 00:00:00
Latest date: 2022-09-28 00:00:00

Number of records by year:


date
2006     214
2007     603
2008     296
2009     456
2010     420
2011     781
2012    1046
2013     424
2014     320
2015     460
2016     532
2017    1321
2018     357
2019     942
2020     987
2021     944
2022    1200
Name: count, dtype: int64

In [11]:
# TEMPORAL CONSISTENCY CHECK

date_year = df_clean["date"].dt.year
date_month = df_clean["date"].dt.month
date_day_of_year = df_clean["date"].dt.dayofyear

year_mismatch = (date_year != df_clean["year"]).sum()
month_mismatch = (date_month != df_clean["month"]).sum()
day_mismatch = (date_day_of_year != df_clean["day_of_year"]).sum()

print("Year mismatches:", year_mismatch)
print("Month mismatches:", month_mismatch)
print("Day-of-year mismatches:", day_mismatch)

Year mismatches: 0
Month mismatches: 0
Day-of-year mismatches: 0


In [12]:
# TEMPORAL CONSISTENCY VERIFICATION

assert year_mismatch == 0
assert month_mismatch == 0
assert day_mismatch == 0

print("Temporal consistency verification passed.")

Temporal consistency verification passed.


In [13]:
# MISSING-VALUE PREPROCESSING SUMMARY

missing_columns = df_clean.columns[
    df_clean.isna().any()
]

missing_summary = pd.DataFrame({
    "missing_count": df_clean[missing_columns].isna().sum(),
    "missing_percentage": (
        df_clean[missing_columns].isna().mean() * 100
    )
}).sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage
temperature_c,1461,12.925772
dew_point_c,1461,12.925772
relative_humidity,1461,12.925772
wind_speed,1461,12.925772
rainfall_mm,1461,12.925772
surface_pressure,1461,12.925772
solar_radiation,1461,12.925772
soil_moisture,1134,10.032735
wind_direction,414,3.662744
lai,185,1.636734


In [14]:
# CHECK WHETHER MISSINGNESS IS CONCENTRATED

weather_missing_cols = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
    "rainfall_mm",
    "surface_pressure",
    "solar_radiation"
]

weather_missing_mask = df_clean[weather_missing_cols].isna().all(axis=1)

print(
    "Records with all major weather variables missing:",
    weather_missing_mask.sum()
)

print(
    "\nPercentage of dataset:",
    round(weather_missing_mask.mean() * 100, 2),
    "%"
)

print("\nMissing records by year:")
display(
    df_clean.loc[weather_missing_mask, "year"]
    .value_counts()
    .sort_index()
)

Records with all major weather variables missing: 414

Percentage of dataset: 3.66 %

Missing records by year:


year
2006    12
2007    33
2008     8
2009    22
2010    13
2011    26
2012    41
2013    14
2014    10
2015    18
2016    26
2017    44
2018    12
2019    24
2020    42
2021    23
2022    46
Name: count, dtype: int64

### Missing Weather Data Interpretation

The dataset contains missing values in several meteorological variables. The
largest amount of missingness occurs in temperature, dew point temperature,
relative humidity, wind speed, rainfall, surface pressure, and solar radiation,
with each variable containing 1,461 missing observations (approximately
12.93% of the dataset).

However, these missing values do not necessarily occur in exactly the same
records. A combined missingness analysis identified 414 records (3.66% of
the dataset) where all major weather variables were simultaneously missing.

The missingness is distributed across the study period rather than being
restricted to a single year. Therefore, the missing values are unlikely to
represent a simple issue affecting one particular year.

Because the weather variables are important predictors of wildfire behaviour,
these records will not be removed solely because of missing weather data.
Instead, missing values will be handled during the preprocessing stage using
a data-driven imputation strategy. The imputation procedure will be fitted
using the training data only to prevent information leakage into the test set.

In [15]:
# TARGET DISTRIBUTION BY WEATHER DATA AVAILABILITY

weather_missing_cols = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
    "rainfall_mm",
    "surface_pressure",
    "solar_radiation"
]

weather_missing = df_clean[weather_missing_cols].isna().any(axis=1)

target_comparison = pd.DataFrame({
    "Weather data status": [
        "Complete weather data",
        "At least one weather variable missing"
    ],
    "Records": [
        (~weather_missing).sum(),
        weather_missing.sum()
    ],
    "Mean burned area (ha)": [
        df_clean.loc[~weather_missing, "burned_area_ha"].mean(),
        df_clean.loc[weather_missing, "burned_area_ha"].mean()
    ],
    "Median burned area (ha)": [
        df_clean.loc[~weather_missing, "burned_area_ha"].median(),
        df_clean.loc[weather_missing, "burned_area_ha"].median()
    ],
    "Maximum burned area (ha)": [
        df_clean.loc[~weather_missing, "burned_area_ha"].max(),
        df_clean.loc[weather_missing, "burned_area_ha"].max()
    ]
})

display(target_comparison)

,Weather data status,Records,Mean burned area (ha),Median burned area (ha),Maximum burned area (ha)
0,Complete weather data,9842,495.939646,132.0,107602.0
1,At least one weather variable missing,1461,444.989733,124.0,54769.0


In [16]:
# Compare log-transformed target

df_clean["log_burned_area"] = np.log1p(df_clean["burned_area_ha"])

print("Mean log burned area:")
print(
    df_clean.groupby(weather_missing)["log_burned_area"].mean()
)

print("\nMedian log burned area:")
print(
    df_clean.groupby(weather_missing)["log_burned_area"].median()
)

Mean log burned area:
False    5.129562
True     5.076381
Name: log_burned_area, dtype: float64

Median log burned area:
False    4.890349
True     4.828314
Name: log_burned_area, dtype: float64


## Interpretation of Missing Weather Data

The relationship between missing weather observations and the target variable
was examined before selecting an imputation strategy.

The dataset contains 9,842 records with complete major weather information and
1,461 records with at least one missing major weather variable. The complete
group has a mean burned area of approximately 495.94 ha and a median of
132 ha, while the missing-weather group has a mean of approximately
444.99 ha and a median of 124 ha.

After applying the log(1 + burned area) transformation, the mean values were
5.13 and 5.08 respectively, while the median values were 4.89 and 4.83.

The relatively small difference between the two groups suggests that missing
weather information is not associated with a substantially different target
distribution. Therefore, the affected observations will be retained rather
than removed.

Missing values will be handled using a model-safe preprocessing strategy.
The dataset will first be divided into training and testing subsets. All
imputation parameters will then be learned exclusively from the training
data and subsequently applied to both the training and testing data to
prevent data leakage.

For continuous numerical variables with substantial missingness, including
temperature, dew point temperature, relative humidity, wind speed, rainfall,
surface pressure, solar radiation, and soil moisture, missing values will be
replaced using the median value calculated from the training data. Median
imputation is preferred because several of these variables have skewed
distributions and may contain extreme observations.

Wind direction will be treated separately because it represents circular
data. It will be transformed into sine and cosine components so that the
cyclic relationship between directions such as 359° and 0° is preserved.

Variables with smaller amounts of missing data, including LAI, NDVI, and
population, will also be handled using training-set median imputation.

Missing-value indicator features will be created for the variables with
meaningful levels of missingness. These indicators will record whether a
value was originally observed or missing before imputation, allowing the
machine learning model to retain potentially useful information about the
missingness pattern.

The target variable, burned_area_ha, contains no missing values and will not
be imputed or modified during this stage.

## Train-Test Split

The cleaned dataset is divided into training and testing subsets before
fitting any imputation or preprocessing parameters.

The training set will be used to learn preprocessing parameters, while the
test set will only be used to evaluate the final machine learning models.

This separation prevents information from the test set from influencing the
preprocessing process and helps prevent data leakage.

An 80/20 split is used, with a fixed random state to ensure reproducibility.

In [17]:
from sklearn.model_selection import train_test_split

# Define target
TARGET = "burned_area_ha"

# Separate predictors and target
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET].copy()

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nTraining percentage:",
      round(len(X_train) / len(df_clean) * 100, 2), "%")

print("Testing percentage:",
      round(len(X_test) / len(df_clean) * 100, 2), "%")

Training set shape: (9042, 32)
Testing set shape: (2261, 32)

Training target shape: (9042,)
Testing target shape: (2261,)

Training percentage: 80.0 %
Testing percentage: 20.0 %


In [ ]:
# IDENTIFY NUMERICAL FEATURES WITH MISSING VALUES

numeric_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

missing_numeric = (
    X_train[numeric_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_numeric = missing_numeric[
    missing_numeric > 0
]

print("Numerical features with missing values:")
display(
    missing_numeric.to_frame("missing_count")
)

Numerical features with missing values:


,missing_count
temperature_c,1157
relative_humidity,1157
dew_point_c,1157
wind_speed,1157
solar_radiation,1157
surface_pressure,1157
rainfall_mm,1157
soil_moisture,903
wind_direction,322
lai,153


In [19]:
# VERIFY TRAIN/TEST MISSING VALUES

print("Missing values in X_train:")
display(
    X_train.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nMissing values in X_test:")
display(
    X_test.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

Missing values in X_train:


temperature_c        1157
wind_speed           1157
relative_humidity    1157
dew_point_c          1157
surface_pressure     1157
solar_radiation      1157
rainfall_mm          1157
soil_moisture         903
wind_direction        322
lai                   153
ndvi                   35
population             12
latitude                0
date                    0
elevation               0
dtype: int64


Missing values in X_test:


temperature_c        304
wind_speed           304
relative_humidity    304
dew_point_c          304
surface_pressure     304
solar_radiation      304
rainfall_mm          304
soil_moisture        231
wind_direction        92
lai                   32
ndvi                   5
population             4
latitude               0
date                   0
elevation              0
dtype: int64

## Missing-Value Imputation Strategy

Numerical variables are grouped according to their preprocessing
requirements. Continuous meteorological and environmental variables will use
median imputation because several distributions are skewed and contain
extreme values.

Wind direction is treated separately because it is circular data. It will
later be represented using sine and cosine transformations.

The imputation values are learned exclusively from the training set and
then applied to the testing set.

In [20]:
# DEFINE IMPUTATION FEATURE GROUPS

# Continuous weather and environmental variables
median_impute_features = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity",
    "wind_speed",
    "rainfall_mm",
    "surface_pressure",
    "solar_radiation",
    "soil_moisture",
    "lai",
    "ndvi",
    "population"
]

# Wind direction will be handled separately because it is circular
wind_direction_feature = "wind_direction"

print("Median-imputed features:")
print(median_impute_features)

print("\nSpecial circular feature:")
print(wind_direction_feature)

Median-imputed features:
['temperature_c', 'dew_point_c', 'relative_humidity', 'wind_speed', 'rainfall_mm', 'surface_pressure', 'solar_radiation', 'soil_moisture', 'lai', 'ndvi', 'population']

Special circular feature:
wind_direction


In [21]:
# CREATE MISSINGNESS INDICATORS

indicator_features = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity",
    "wind_speed",
    "rainfall_mm",
    "surface_pressure",
    "solar_radiation",
    "soil_moisture",
    "wind_direction"
]

for feature in indicator_features:
    X_train[f"{feature}_missing"] = X_train[feature].isna().astype(int)
    X_test[f"{feature}_missing"] = X_test[feature].isna().astype(int)

print("Missingness indicators created:")
print(
    [f"{feature}_missing" for feature in indicator_features]
)

Missingness indicators created:
['temperature_c_missing', 'dew_point_c_missing', 'relative_humidity_missing', 'wind_speed_missing', 'rainfall_mm_missing', 'surface_pressure_missing', 'solar_radiation_missing', 'soil_moisture_missing', 'wind_direction_missing']


In [22]:
# FIT MEDIAN IMPUTER USING TRAINING DATA ONLY

from sklearn.impute import SimpleImputer

median_imputer = SimpleImputer(strategy="median")

# Learn medians ONLY from training data
median_imputer.fit(
    X_train[median_impute_features]
)

print("Median imputer fitted using training data only.")

print("\nLearned median values:")
display(
    pd.Series(
        median_imputer.statistics_,
        index=median_impute_features,
        name="training_median"
    )
)

Median imputer fitted using training data only.

Learned median values:


temperature_c        2.921554e+01
dew_point_c          1.392626e+01
relative_humidity    3.030152e+01
wind_speed           3.154953e+00
rainfall_mm          3.255904e-03
surface_pressure     9.546614e+04
solar_radiation      1.325231e+07
soil_moisture        2.862000e-01
lai                  1.000000e+00
ndvi                 4.658000e-01
population           5.372214e+00
Name: training_median, dtype: float64

In [23]:
# APPLY MEDIAN IMPUTATION

X_train[median_impute_features] = median_imputer.transform(
    X_train[median_impute_features]
)

X_test[median_impute_features] = median_imputer.transform(
    X_test[median_impute_features]
)

print("Median imputation completed.")

Median imputation completed.


In [24]:
# VERIFY REMAINING MISSING VALUES

print("Remaining missing values in X_train:")
display(
    X_train.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

print("\nRemaining missing values in X_test:")
display(
    X_test.isna().sum()
    .sort_values(ascending=False)
    .head(15)
)

Remaining missing values in X_train:


wind_direction       322
date                   0
latitude               0
temperature_c          0
longitude              0
dew_point_c            0
relative_humidity      0
wind_speed             0
rainfall_mm            0
surface_pressure       0
solar_radiation        0
ndvi                   0
lai                    0
soil_moisture          0
elevation              0
dtype: int64


Remaining missing values in X_test:


wind_direction       92
date                  0
latitude              0
temperature_c         0
longitude             0
dew_point_c           0
relative_humidity     0
wind_speed            0
rainfall_mm           0
surface_pressure      0
solar_radiation       0
ndvi                  0
lai                   0
soil_moisture         0
elevation             0
dtype: int64

## Circular Transformation of Wind Direction

Wind direction is a circular variable measured in degrees. Directly treating
wind direction as a linear numerical variable can introduce an artificial
discontinuity between 0° and 360°.

Therefore, wind direction is converted into sine and cosine components.
Missing wind-direction observations are handled using a training-derived
circular representation.

In [25]:
# HANDLE WIND DIRECTION AS A CIRCULAR FEATURE

# Convert degrees to radians
wind_train_rad = np.deg2rad(X_train["wind_direction"])
wind_test_rad = np.deg2rad(X_test["wind_direction"])

# Create circular components
X_train["wind_direction_sin"] = np.sin(wind_train_rad)
X_train["wind_direction_cos"] = np.cos(wind_train_rad)

X_test["wind_direction_sin"] = np.sin(wind_test_rad)
X_test["wind_direction_cos"] = np.cos(wind_test_rad)

print("Wind direction converted to circular features.")

Wind direction converted to circular features.


In [26]:
# IMPUTE CIRCULAR WIND-DIRECTION COMPONENTS

wind_direction_imputer = SimpleImputer(strategy="median")

# Fit using training data only
wind_direction_imputer.fit(
    X_train[["wind_direction_sin", "wind_direction_cos"]]
)

# Apply to train and test
X_train[["wind_direction_sin", "wind_direction_cos"]] = (
    wind_direction_imputer.transform(
        X_train[["wind_direction_sin", "wind_direction_cos"]]
    )
)

X_test[["wind_direction_sin", "wind_direction_cos"]] = (
    wind_direction_imputer.transform(
        X_test[["wind_direction_sin", "wind_direction_cos"]]
    )
)

print("Circular wind-direction imputation completed.")

Circular wind-direction imputation completed.


In [27]:
# VERIFY WIND-DIRECTION PROCESSING

print("Remaining missing values:")
display(
    X_train[[
        "wind_direction",
        "wind_direction_sin",
        "wind_direction_cos"
    ]].isna().sum()
)

print("\nTest set:")
display(
    X_test[[
        "wind_direction",
        "wind_direction_sin",
        "wind_direction_cos"
    ]].isna().sum()
)

Remaining missing values:


wind_direction        322
wind_direction_sin      0
wind_direction_cos      0
dtype: int64


Test set:


wind_direction        92
wind_direction_sin     0
wind_direction_cos     0
dtype: int64

In [28]:
# REMOVE ORIGINAL WIND DIRECTION

X_train = X_train.drop(columns=["wind_direction"])
X_test = X_test.drop(columns=["wind_direction"])

print("Original wind_direction removed.")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Original wind_direction removed.
X_train shape: (9042, 42)
X_test shape: (2261, 42)


In [30]:
# FINAL MISSING-VALUE CHECK

print("Total missing values in X_train:")
print(X_train.isna().sum().sum())

print("\nTotal missing values in X_test:")
print(X_test.isna().sum().sum())

Total missing values in X_train:
0

Total missing values in X_test:
0


In [31]:
print("Training features:")
print(X_train.columns.tolist())

print("\nNumber of training features:")
print(len(X_train.columns))

print("\nTesting features:")
print(X_test.columns.tolist())

Training features:
['date', 'latitude', 'longitude', 'temperature_c', 'dew_point_c', 'relative_humidity', 'wind_speed', 'rainfall_mm', 'surface_pressure', 'solar_radiation', 'ndvi', 'lai', 'soil_moisture', 'elevation', 'slope_degrees', 'aspect', 'curvature', 'roads_distance_km', 'population', 'lc_agriculture', 'lc_forest', 'lc_grassland', 'lc_settlement', 'lc_shrubland', 'lc_sparse_vegetation', 'lc_water_bodies', 'lc_wetland', 'year', 'month', 'day_of_year', 'log_burned_area', 'temperature_c_missing', 'dew_point_c_missing', 'relative_humidity_missing', 'wind_speed_missing', 'rainfall_mm_missing', 'surface_pressure_missing', 'solar_radiation_missing', 'soil_moisture_missing', 'wind_direction_missing', 'wind_direction_sin', 'wind_direction_cos']

Number of training features:
42

Testing features:
['date', 'latitude', 'longitude', 'temperature_c', 'dew_point_c', 'relative_humidity', 'wind_speed', 'rainfall_mm', 'surface_pressure', 'solar_radiation', 'ndvi', 'lai', 'soil_moisture', 'elevat

## Target and Predictor Separation

The transformed burned-area variable, `log_burned_area`, is the prediction
target. It must not be included among the predictor variables because doing so
would cause target leakage.

The target is therefore separated from both training and testing feature
sets before further preprocessing.

In [32]:
# REMOVE TARGET FROM FEATURE SETS

target_column = "log_burned_area"

y_train = X_train[target_column].copy()
y_test = X_test[target_column].copy()

X_train = X_train.drop(columns=[target_column])
X_test = X_test.drop(columns=[target_column])

print("Target separated successfully.")

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

print("\nTraining feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

Target separated successfully.

Training target shape: (9042,)
Testing target shape: (2261,)

Training feature shape: (9042, 41)
Testing feature shape: (2261, 41)


In [33]:
# VERIFY TARGET LEAKAGE

print("Is target in X_train?", target_column in X_train.columns)
print("Is target in X_test?", target_column in X_test.columns)

assert target_column not in X_train.columns
assert target_column not in X_test.columns

print("\nTarget leakage check passed.")

Is target in X_train? False
Is target in X_test? False

Target leakage check passed.


## Temporal Feature Engineering

Temporal variables contain cyclical patterns that are not fully represented by
their raw numerical values.

Month and day-of-year are therefore transformed using sine and cosine
encoding. This allows the model to represent the circular nature of seasonal
patterns, where the end of one cycle is naturally connected to the beginning
of the next.

The original year variable is retained to capture longer-term temporal
variation.

In [34]:
# TEMPORAL CYCLICAL FEATURES

# Month cycle: 1–12
X_train["month_sin"] = np.sin(
    2 * np.pi * X_train["month"] / 12
)

X_train["month_cos"] = np.cos(
    2 * np.pi * X_train["month"] / 12
)

X_test["month_sin"] = np.sin(
    2 * np.pi * X_test["month"] / 12
)

X_test["month_cos"] = np.cos(
    2 * np.pi * X_test["month"] / 12
)


# Day-of-year cycle: approximately 365 days
X_train["day_of_year_sin"] = np.sin(
    2 * np.pi * X_train["day_of_year"] / 365
)

X_train["day_of_year_cos"] = np.cos(
    2 * np.pi * X_train["day_of_year"] / 365
)

X_test["day_of_year_sin"] = np.sin(
    2 * np.pi * X_test["day_of_year"] / 365
)

X_test["day_of_year_cos"] = np.cos(
    2 * np.pi * X_test["day_of_year"] / 365
)

print("Cyclical temporal features created.")

Cyclical temporal features created.


In [35]:
temporal_features = [
    "month_sin",
    "month_cos",
    "day_of_year_sin",
    "day_of_year_cos"
]

display(
    X_train[temporal_features].describe()
)

,month_sin,month_cos,day_of_year_sin,day_of_year_cos
count,9.042000e+03,9.042000e+03,9042.000000,9042.000000
mean,-3.925978e-01,-3.950533e-01,-0.279367,-0.490389
std,6.713461e-01,4.890507e-01,0.640377,0.521024
min,-1.000000e+00,-1.000000e+00,-0.999991,-0.999963
25%,-8.660254e-01,-8.660254e-01,-0.785650,-0.912375
50%,-5.000000e-01,-5.000000e-01,-0.486273,-0.683919
75%,1.224647e-16,-1.836970e-16,0.032271,-0.183998
max,1.000000e+00,1.000000e+00,0.999991,1.000000


In [36]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (9042, 45)
Testing shape: (2261, 45)


## Temporal Feature Engineering

Temporal variables were processed to capture the seasonal and cyclical nature of wildfire activity. The original `month` and `day_of_year` variables were transformed using sine and cosine encoding.

The following cyclical features were created:

- `month_sin`
- `month_cos`
- `day_of_year_sin`
- `day_of_year_cos`

Cyclical encoding allows the model to understand that the beginning and end of a temporal cycle are closely related. For example, December and January are adjacent months in the annual cycle, even though their numerical values are 12 and 1.

The transformations were calculated as:

$$
month\_sin = \sin\left(\frac{2\pi \times month}{12}\right)
$$

$$
month\_cos = \cos\left(\frac{2\pi \times month}{12}\right)
$$

For the day of year:

$$
day\_sin = \sin\left(\frac{2\pi \times day\_of\_year}{365}\right)
$$

$$
day\_cos = \cos\left(\frac{2\pi \times day\_of\_year}{365}\right)
$$

The resulting features were verified to have values within the expected range of **-1 to +1**, with no missing values.

After temporal feature engineering:

- **Training records:** 9,042
- **Testing records:** 2,261
- **Total predictor features:** 45

The original `year`, `month`, `day_of_year`, and `date` variables were retained temporarily and will be evaluated during the final feature-selection stage. The newly created cyclical features provide the model with additional information about seasonal wildfire patterns.

## Land-Cover Feature Verification

The land-cover variables represent the fractional coverage of different land-cover classes at each observation location.

The following features are available:

- `lc_agriculture`
- `lc_forest`
- `lc_grassland`
- `lc_settlement`
- `lc_shrubland`
- `lc_sparse_vegetation`
- `lc_water_bodies`
- `lc_wetland`

Previous exploratory analysis showed that these variables represent proportions between 0 and 1 and collectively sum to approximately 1 for every observation.

Therefore, they are treated as continuous numerical features rather than categorical variables. No one-hot encoding or additional normalization is required.

All land-cover features will initially be retained so that their predictive contribution can be evaluated during model development.

In [37]:
# LAND-COVER FEATURE VERIFICATION

land_cover_features = [
    "lc_agriculture",
    "lc_forest",
    "lc_grassland",
    "lc_settlement",
    "lc_shrubland",
    "lc_sparse_vegetation",
    "lc_water_bodies",
    "lc_wetland"
]

print("Land-cover features:")
print(land_cover_features)

print("\nMissing values:")
display(
    X_train[land_cover_features].isna().sum()
)

print("\nMinimum values:")
display(
    X_train[land_cover_features].min()
)

print("\nMaximum values:")
display(
    X_train[land_cover_features].max()
)

Land-cover features:
['lc_agriculture', 'lc_forest', 'lc_grassland', 'lc_settlement', 'lc_shrubland', 'lc_sparse_vegetation', 'lc_water_bodies', 'lc_wetland']

Missing values:


lc_agriculture          0
lc_forest               0
lc_grassland            0
lc_settlement           0
lc_shrubland            0
lc_sparse_vegetation    0
lc_water_bodies         0
lc_wetland              0
dtype: int64


Minimum values:


lc_agriculture          0.0
lc_forest               0.0
lc_grassland            0.0
lc_settlement           0.0
lc_shrubland            0.0
lc_sparse_vegetation    0.0
lc_water_bodies         0.0
lc_wetland              0.0
dtype: float64


Maximum values:


lc_agriculture          1.000000
lc_forest               1.000000
lc_grassland            1.000000
lc_settlement           0.890119
lc_shrubland            1.000000
lc_sparse_vegetation    1.000000
lc_water_bodies         1.000000
lc_wetland              1.000000
dtype: float64

In [39]:
# VERIFY LAND-COVER SUM

train_land_cover_total = X_train[land_cover_features].sum(axis=1)
test_land_cover_total = X_test[land_cover_features].sum(axis=1)

print("Training land-cover total:")
print(train_land_cover_total.describe())

print("\nTesting land-cover total:")
print(test_land_cover_total.describe())

print(
    "\nTraining records not approximately equal to 1:",
    (~np.isclose(
        train_land_cover_total,
        1.0,
        atol=0.01
    )).sum()
)

print(
    "Testing records not approximately equal to 1:",
    (~np.isclose(
        test_land_cover_total,
        1.0,
        atol=0.01
    )).sum()
)

Training land-cover total:
count    9.042000e+03
mean     1.000000e+00
std      1.869280e-08
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

Testing land-cover total:
count    2.261000e+03
mean     1.000000e+00
std      1.879100e-08
min      9.999999e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

Training records not approximately equal to 1: 0
Testing records not approximately equal to 1: 0


## Final Predictor Feature Selection

Following data cleaning, missing-value handling, circular encoding, temporal
feature engineering, and land-cover verification, the final predictor
variables are defined.

The raw `date` variable is removed because its useful temporal information has
already been represented through `year`, `month`, `day_of_year`, and cyclical
sine/cosine features.

The final predictor set contains spatial, topographical, meteorological,
environmental, land-cover, temporal, missingness-indicator, and wind-direction
features.

The transformed burned-area variable `log_burned_area` is maintained
separately as the prediction target and is not included among the predictors.